In [ ]:
# House Prices - Advanced Regression Solution
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

# Save IDs
train_ID = train['Id']
test_ID = test['Id']

# Remove IDs
train.drop('Id', axis=1, inplace=True)
test.drop('Id', axis=1, inplace=True)

# Check target variable
print(f"\nTarget variable statistics:")
print(train['SalePrice'].describe())

# Feature Engineering
def engineer_features(df, is_train=True):
    data = df.copy()
    
    # Log transform for SalePrice (target) - only for training
    if is_train and 'SalePrice' in data.columns:
        data['SalePrice'] = np.log1p(data['SalePrice'])
    
    # Total square footage
    data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']
    
    # Total bathrooms
    data['TotalBath'] = (data['FullBath'] + 
                         0.5 * data['HalfBath'] + 
                         data['BsmtFullBath'] + 
                         0.5 * data['BsmtHalfBath'])
    
    # Total porch area
    data['TotalPorchSF'] = (data['OpenPorchSF'] + 
                            data['3SsnPorch'] + 
                            data['EnclosedPorch'] + 
                            data['ScreenPorch'] + 
                            data['WoodDeckSF'])
    
    # House age
    data['HouseAge'] = data['YrSold'] - data['YearBuilt']
    data['RemodAge'] = data['YrSold'] - data['YearRemodAdd']
    
    # Has pool, garage, basement, fireplace
    data['HasPool'] = (data['PoolArea'] > 0).astype(int)
    data['HasGarage'] = (data['GarageArea'] > 0).astype(int)
    data['HasBsmt'] = (data['TotalBsmtSF'] > 0).astype(int)
    data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)
    data['Has2ndFloor'] = (data['2ndFlrSF'] > 0).astype(int)
    
    # Quality scores
    data['OverallScore'] = data['OverallQual'] * data['OverallCond']
    data['GarageScore'] = data['GarageQual'].fillna('None').map(
        {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    ).fillna(0) * data['GarageCond'].fillna('None').map(
        {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    ).fillna(0)
    
    # Quality interactions
    data['QualArea'] = data['OverallQual'] * data['GrLivArea']
    data['QualBath'] = data['OverallQual'] * data['TotalBath']
    
    return data

print("\nEngineering features...")
train = engineer_features(train, is_train=True)
test = engineer_features(test, is_train=False)

# Separate target
y = train['SalePrice'].copy()
train.drop('SalePrice', axis=1, inplace=True)

# Handle missing values
def handle_missing(df):
    data = df.copy()
    
    # Categorical: fill with 'None'
    categorical_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                        'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                        'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                        'MasVnrType']
    for col in categorical_none:
        if col in data.columns:
            data[col].fillna('None', inplace=True)
    
    # Numerical: fill with 0
    numerical_zero = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                      'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
                      'GarageYrBlt', 'GarageArea', 'GarageCars']
    for col in numerical_zero:
        if col in data.columns:
            data[col].fillna(0, inplace=True)
    
    # LotFrontage: fill with median by Neighborhood
    if 'LotFrontage' in data.columns:
        data['LotFrontage'] = data.groupby('Neighborhood')['LotFrontage'].transform(
            lambda x: x.fillna(x.median())
        )
    
    # Fill remaining with mode or median
    for col in data.columns:
        if data[col].dtype == 'object':
            data[col].fillna(data[col].mode()[0] if not data[col].mode().empty else 'None', inplace=True)
        else:
            data[col].fillna(data[col].median(), inplace=True)
    
    return data

print("\nHandling missing values...")
train = handle_missing(train)
test = handle_missing(test)

# Get numeric and categorical features
numeric_features = train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Encode categorical variables
train_encoded = pd.get_dummies(train, columns=categorical_features, drop_first=True)
test_encoded = pd.get_dummies(test, columns=categorical_features, drop_first=True)

# Align columns
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

print(f"\nFinal feature shape: {train_encoded.shape}")

# Scale features
scaler = RobustScaler()
X_scaled = scaler.fit_transform(train_encoded)
X_test_scaled = scaler.transform(test_encoded)

# Models
print("\nTraining models...")

# Model 1: Ridge Regression
ridge = Ridge(alpha=10.0, random_state=42)

# Model 2: Lasso Regression
lasso = Lasso(alpha=0.0005, random_state=42)

# Model 3: Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Model 4: Gradient Boosting
gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

models = [
    ('Ridge', ridge, X_scaled),
    ('Lasso', lasso, X_scaled),
    ('Random Forest', rf, train_encoded.values),
    ('Gradient Boosting', gb, train_encoded.values)
]

print("\nCross-Validation RMSE:")
cv_scores = {}
for name, model, X_data in models:
    scores = cross_val_score(model, X_data, y, 
                            scoring='neg_root_mean_squared_error', 
                            cv=kfold)
    rmse = -scores.mean()
    cv_scores[name] = rmse
    print(f"{name:20} - RMSE: {rmse:.4f} (+/- {scores.std():.4f})")

# Train all models and make ensemble
print("\nTraining final ensemble...")
predictions = {}

for name, model, X_data in models:
    model.fit(X_data, y)
    if name in ['Ridge', 'Lasso']:
        pred = model.predict(X_test_scaled)
    else:
        pred = model.predict(test_encoded.values)
    predictions[name] = pred

# Weighted ensemble based on CV performance
weights = {name: 1/cv_scores[name] for name in cv_scores}
total_weight = sum(weights.values())
weights = {name: w/total_weight for name, w in weights.items()}

print("\nModel Weights:")
for name, weight in weights.items():
    print(f"{name:20} - Weight: {weight:.3f}")

# Final prediction (weighted average)
final_pred = sum(predictions[name] * weights[name] for name in predictions)

# Convert back from log scale
final_pred = np.expm1(final_pred)

# Create submission
submission = pd.DataFrame({
    'Id': test_ID,
    'SalePrice': final_pred
})

submission.to_csv('submission.csv', index=False)

print("\nSubmission created!")
print(f"Predicted prices range: ${final_pred.min():.0f} - ${final_pred.max():.0f}")
print(f"Mean predicted price: ${final_pred.mean():.0f}")

# Feature importance from Random Forest
rf.fit(train_encoded.values, y)
importance = pd.DataFrame({
    'Feature': train_encoded.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Important Features:")
print(importance.head(15).to_string(index=False))

print("\n" + "="*60)
print("Done! Expected RMSE: 0.12-0.14")
print("="*60)